In [1]:
import socket
import threading
import json

In [2]:
# -----------------------------------
# 1. 서버 접속 설정
# -----------------------------------
HOST = '192.168.0.46' # 서버 아이피 주소
PORT = 58129          # 서버 포트 번호 (서버 코드와 동일해야 함)
MAX_CLIENTS = 10 

In [ ]:
# -----------------------------------
# 2. 분석 함수 정의
#------------------------------------
def analyze_text(request):
    mode = request.get("mode")
    text = request.get("text", "")

    if mode == "length":
        return {
            "result": len(text),
            "desc": f"문자 길이는 {len(text)}자 입니다."  # f-string 안에 [] 말고 {} 써야해요!
        }
    elif mode == "sentiment":
        if any(word in text for word in ["좋아", "행복", "멋져", "훌륭", "사랑"]):
            sentiment = "positive"
        elif any(word in text for word in ["싫어", "불만", "짜증", "나빠", "불행"]):
            sentiment = "negative"
        else:
            sentiment = "neutral"
        return {
            "result": sentiment,
            "desc": f"감정 분석 결과: {sentiment}"
        }
    elif mode == "keywords":
        keywords = ["AI", "고장", "불량", "스크래치", "찍힘"]
        found = [k for k in keywords if k in text]
        return {
            "result": found,
            "desc": f"키워드 발견: {', '.join(found) if found else '없음'}"
        }
    else:
        return {"error": f"지원하지 않는 분석 모드입니다. : {mode}"}

# -----------------------------------
# 3. 클라이언트 처리 스레드 함수
#------------------------------------
def handle_client(client_socket, address):
    """
    각 클라이언트 연결마다 실행되는 함수
    : param cient_socket : 해당 클라이언트의 소켓
    : param address : 클라이언트의 주소 정보
    """
    print(f" 클라이언트 {address} 연결됨")

    while True:
        try:
            data= client_socket.recv(2048).decode()   # 클라이언트로부터 데이터 수집

            if not data:   # 클라이언트 연결 종료 감지
                print(f"{address} 연결 끊김")
                break

            if data.lower() == "exit":
                print(f"{address} 종료 요청 수신")
                break

            # json data 파싱
            try:
                request = json.loads(data)
                result = analyze_text(request)
            except json.JSONDecodeError:
                result = {"error": "잘못된 json 형식입니다."}

            # 응답 전송 (json -> bytes)
            response = json.dumps(result, ensure_ascii=False)
            client_socket.sendall(response.encode())

        except ConnectionResetError:
            print(f" {address} 비정상 종료")
            break

        client_socket.close()
        print(f" 클라이언트 {address} 세션 종료 완료")

# -----------------------------------
# 4. 서버 메인 실행부
#------------------------------------
def start_server():
    """
    메인 서버 함수 : 클라이언트 접속을 대기하고,
    접속 시마다 새로운 쓰레드를 생성하여 handle_client 실행
    """
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.bind((HOST, PORT))
    server_socket.listen(MAX_CLIENTS)

    print(f" AI 서버 실행 중.... {HOST}:{PORT}")
    print(f" 최대 {MAX_CLIENTS} 개의 클라이언트 동시 접속 가능\n")

    try:
        while True:
            client_socket, addr = server_socket.accept()

            client_thread = threading.Thread(
            target = handle_client, args=(client_socket, addr), daemon=True
            )
            client_thread.start()
    except KeyboardInterrupt:
        print("\n 서버 수동 종료 감지")
    finally:
        server_socket.close()
        print(" 서버 완전 종료")

# -----------------------------------
# 5. 실행 시작
#------------------------------------
if __name__ == "__main__":
    start_server()

 AI 서버 실행 중.... 192.168.0.46:58129
 최대 10 개의 클라이언트 동시 접속 가능

 클라이언트 ('192.168.0.46', 50885) 연결됨


Exception in thread Thread-3 (handle_client):
Traceback (most recent call last):
  File "C:\Users\ui203\anaconda3\envs\aigugbi\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "C:\Users\ui203\anaconda3\envs\aigugbi\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\ui203\AppData\Local\Temp\ipykernel_10900\226579874.py", line 47, in handle_client
OSError: [WinError 10038] 소켓 이외의 개체에 작업을 시도했습니다


 클라이언트 ('192.168.0.46', 50885) 세션 종료 완료
